In [1]:
import os
import numpy as np
import time
import pandas as pd
from scipy import interpolate
import pickle
import sys
from astropy.io import ascii
from astropy.table import Table
from astropy import units as u
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap

In [15]:
def Enquiry(HashTable, InfoDict, Band1, Band2, dT1, dT2, dMag=None, Color=None):
    if abs(dT1) > abs(dT1-dT2):
        dT1, dT2 = dT1-dT2, -dT2    
    Ind1 = InfoDict['BandPairs'].index(Band1+Band2)
    #index for where 1st filter is, 2nd index for 2nd filter
    dT1grid = InfoDict['dT1s'][ abs( dT1 - InfoDict['dT1s'] ).argmin() ]
    dT2grid = InfoDict['dT2s'][ abs( dT2 - InfoDict['dT2s'] ).argmin() ]
    
    TimePairGrid = np.array([ InfoDict['dT1s'][ abs( dT1 - InfoDict['dT1s'] ).argmin() ], InfoDict['dT2s'][ abs( dT2 - InfoDict['dT2s'] ).argmin() ] ])
    # above will need some difference to set limit on difference in times
    Ind2 = np.where( (np.array(InfoDict['TimePairs']) == TimePairGrid ).all(axis=1) )[0][0]
    Results = HashTable[Ind1, Ind2]

    if dMag == None:
        pass        
    elif dMag<InfoDict['BinMag'][0] or dMag>=InfoDict['BinMag'][-1]:
        raise ValueError('The value of dMag is out of boundary, the available interval is [{:.2f}, {:.2f}).'.format(InfoDict['BinMag'][0], InfoDict['BinMag'][-1]))        
    else:
        Results = Results[np.where( dMag >= InfoDict['BinMag'] )[0][-1]]       

    if Color == None:
        pass        
    elif Color<InfoDict['BinColor'][0] or Color>=InfoDict['BinColor'][-1]:
        raise ValueError('The value of Color is out of boundary, the available interval is [{:.2f}, {:.2f}).'.format(InfoDict['BinColor'][0], InfoDict['BinColor'][-1]))
        
    else:
        Results = Results[..., np.where( Color >= InfoDict['BinColor'] )[0][-1] ]

    return Results

In [16]:
def loadCubeFile(FilePath):
    
    FileTime = FilePath[FilePath.find('Cube_')+4: FilePath.rfind('__')]
    
    with open(FilePath, 'rb') as f:
        InfoDict = pickle.load(f)
        print(InfoDict)
        HashTable = pickle.load(f)
    
    dT1Range = [InfoDict['dT1s'][0], InfoDict['dT1s'][-1]]
    dT2Range = [InfoDict['dT2s'][0], InfoDict['dT2s'][-1]]    
    dT1step = ( InfoDict['dT1s'][1:] - InfoDict['dT1s'][:-1] ).min()
    dT2step = ( InfoDict['dT2s'][1:] - InfoDict['dT2s'][:-1] ).min()
        
    StartObjNo = 'n/a'
    
    if 'StartObjNo' in InfoDict:
        StartObjNo = InfoDict['StartObjNo']
        
   # print('{:<35}ObjectNo: {:>5}, start at {:>3}. dT1 range = {}, step = {:>3}. dT2 range = {}, step = {:>3}. BandpairNo: {}.'.format(
   #     InfoDict['EventNames']+FileTime, InfoDict['ObjectNo'], StartObjNo, dT1Range, dT1step, dT2Range, dT2step, len(InfoDict['BandPairs'])))

    # InfoDict['OutliersRatio'] = InfoDict['Outliers'] / HashTable.sum()

    if 'Outliers' in InfoDict:
        print('\t{} outliers found, the ratio to the max value is {:.12f}.'.format(InfoDict['Outliers'], InfoDict['OutliersRatio']) )
        print('\tdMag range is {}, \n\tColor range is {}.'.format( InfoDict['dMagRange'], InfoDict['ColorRange'] ) )

    if 'Overflow' in InfoDict:
        print('\tData in the HashTable overflowed, the minimun value is {}.'.format(InfoDict['Overflow']))
        
    return InfoDict, HashTable;

In [17]:
def PlotSlice(HashTable1, InfoDict1, Band1, Band2, dT1, dT2, ax = None):
    # cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))
    
    if ax == None: 
        fig, ax = plt.subplots(1,1)

    

    Map1 = Enquiry(HashTable1, InfoDict1, Band1, Band2, dT1, dT2) * 10000000

    ax.pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map1)+1,
                        norm=LogNorm(1, vmax=Map1.max()+1), cmap="gist_grey")

        # ax.scatter(Data[0], Data[1], c='mediumpurple', s=1, alpha=0.1, )


    ax.set_xlim([-1.5, 2])
    ax.set_ylim([-5, 8]) 


In [18]:
def PlotSliceWithTwoHists(HashTable1, InfoDict1, HashTable2, InfoDict2, Band1, Band2, dT1, dT2, ax = None):
    cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))
    
    if ax == None: 
        fig, ax = plt.subplots(1,1)
    Map1 = Enquiry(HashTable1, InfoDict1, Band1, Band2, dT1, dT2) * 10000000
    Map2 = Enquiry(HashTable2, InfoDict2, Band1, Band2, dT1, dT2) * 10000000
    
    ax.pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map1)+1,
                       norm=LogNorm(1, vmax=Map1.max()+1), cmap='gist_gray')
    ax.pcolor(InfoDict2['BinMag'], InfoDict2['BinColor'], np.transpose(Map2),
                       norm=LogNorm(1, vmax=Map2.max()-1), cmap=cut_gist_heat)
            
    ax.set_xlim([-1.5, 2])
    ax.set_ylim([-5, 8]) 

In [2]:
dp2 = ascii.read("/lustre/lrspec/users/4300/cube/Data/Phot/DP2query1.ecsv")

In [4]:
dp2.info

<Table length=707648>
     name       dtype  unit                                                description                                               
-------------- ------- ---- ---------------------------------------------------------------------------------------------------------
   diaObjectId   int64                                             Id of the DiaObject that this DiaForcedSource was associated with.
          band    str1                                            Abstract filter that is not associated with a particular instrument
       psfFlux float32  nJy                              Flux derived from linear least-squares fit of psf model forced on the calexp
    psfFluxErr float32  nJy           Uncertainty on the flux derived from linear least-squares fit of psf model forced on the calexp
   psfDiffFlux float32  nJy                    Flux derived from linear least-squares fit of psf model forced on the image difference
psfDiffFluxErr float32  nJy Uncertainty 

In [6]:
len(dp2[dp2["psfFlux"]>0])

536326

In [19]:
dp2 = dp2[dp2["psfFlux"]>0]

In [20]:
len(np.unique(dp2["diaObjectId"]))

2774

In [38]:
sel_objs = np.unique(dp2["diaObjectId"])[0:5]
dp2_sel = dp2[np.isin(dp2["diaObjectId"],sel_objs)]

In [35]:
pref_dT1s = np.arange(-480, 481, 15)
pref_dT2s = np.hstack(( np.arange(-1920, -1439, 30), np.arange(-480, 481, 30), np.arange(1440, 1921, 30) )) 
thrs = {'u': 23.9, 'g': 25.0, 'r': 24.7, 'i': 24.0, 'z': 23.3, 'y': 22.1}

In [39]:
obj_dfs = []

for n, obj in enumerate(np.unique(dp2_sel["diaObjectId"])):
    timepairs = []
    bandpairs = []
    dMags = []
    colors = []
    name = []
    dp2_obj = dp2_sel[dp2_sel["diaObjectId"] == obj]
    
    for i,mjd1 in enumerate(dp2_obj["expMidptMJD"]):
        dT1s2 = []
        dT2s2 = []
        dMs = []
        bps = []
        cs = []
        for j, mjd2 in enumerate(dp2_obj["expMidptMJD"]):
            if dp2_obj["band"][i] != dp2_obj["band"][j]:
                dT1 = -(mjd1-mjd2)*u.day
                dT1 = int((dT1.to(u.min)/u.min))

                
                # if dT1 in pref_dT1s:
                M1 = -2.5*np.log10(dp2_obj["psfFlux"][i])+31.4
                M2 = -2.5*np.log10(dp2_obj["psfFlux"][j])+31.4
                color = M1-M2
                #= dp1_obj["psfFlux"][i] / dp1_obj["psfFlux"][j]
                BandPair = dp2_obj["band"][i]+dp2_obj["band"][j]
                    
                
                dT1s2.append(dT1)
                cs.append(color)
                bps.append(BandPair)
            
            else:
                dT2 = -(mjd1-mjd2)*u.day
                dT2 = int((dT2.to(u.min)/u.min))

                
                # if dT2 in pref_dT2s and dT2 != 0:
                M1 = -2.5*np.log10(dp2_obj["psfFlux"][i])+31.4
                M2 = -2.5*np.log10(dp2_obj["psfFlux"][j])+31.4
                
                # dflux = (dp1_obj["psfFlux"][i] / dp1_obj["psfFlux"][j])

                dT2s2.append(dT2)
                #dMs.append((-2.5*np.log10(dflux))*np.sign(dT2))
                dMs.append((M1-M2)*np.sign(dT2))
            
        for k, dT1 in enumerate(dT1s2):
            for h, dT2 in enumerate(dT2s2):
                timepairs.append((dT1,dT2))

                bandpairs.append(bps[k])
                colors.append(cs[k])
                
                dMags.append(dMs[h])

                name.append(obj)
                    

    obj_pts = pd.DataFrame()
    obj_pts["Timepairs"] = timepairs
    obj_pts["dMags"] = dMags
    obj_pts["Colors"] = colors
    obj_pts["Bandpairs"] = bandpairs
    obj_pts["Object Name"] = name

    obj_dfs.append(obj_pts)

    print(f'{n+1} of {len(np.unique(dp2_sel["diaObjectId"]))}')
print("Done!")

1 of 5
2 of 5
3 of 5
4 of 5
5 of 5
Done!


In [42]:
for obj in obj_dfs:
    print(obj["Timepairs"])

0               (5721, 0)
1               (5721, 1)
2               (5721, 0)
3              (5721, -1)
4               (5721, 0)
                ...      
3076835    (53069, 25859)
3076836    (53069, 25895)
3076837    (53069, 25896)
3076838    (53069, 47515)
3076839        (53069, 0)
Name: Timepairs, Length: 3076840, dtype: object
0                 (5721, 0)
1                 (5721, 1)
2                 (5721, 0)
3                (5721, -1)
4                 (5721, 0)
                 ...       
2863123    (-82043, -71965)
2863124    (-82043, -24604)
2863125    (-82043, -21655)
2863126    (-82043, -21620)
2863127         (-82043, 0)
Name: Timepairs, Length: 2863128, dtype: object
0                 (5721, 0)
1                 (5721, 1)
2                 (5721, 0)
3                (5721, -1)
4                 (5721, 0)
                 ...       
2863123    (-82043, -71965)
2863124    (-82043, -24604)
2863125    (-82043, -21655)
2863126    (-82043, -21620)
2863127         (-82043, 0)
Na